# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
# ============================================================
# FIX HUGGING FACE AUTHENTICATION
# ============================================================

!pip -q install -U huggingface_hub fsspec

from huggingface_hub import login

print("Paste your Hugging Face token when prompted.")
login()

Paste your Hugging Face token when prompted.


In [4]:
# ============================================================
# TEST HUGGING FACE ACCESS
# ============================================================

from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

print("Authentication successful.")
print("Downloaded/accessed:")
print(test_file)

Authentication successful.
Downloaded/accessed:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
# ============================================================
# DUCKDB + AUTHENTICATED HUGGING FACE
# ============================================================

import duckdb
import os

from huggingface_hub import hf_hub_download

# Download the exact March and April files through the
# authenticated Hugging Face client.

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset"
)

print("March file:", march_file)
print("April file:", april_file)

con = duckdb.connect()

# ------------------------------------------------------------
# READ MARCH
# ------------------------------------------------------------

df = con.execute(
    f"""
    SELECT *
    FROM read_parquet('{march_file}')
    """
).df()

print("\nMarch rows:", len(df))
print("March columns:", len(df.columns))

display(df.head())

# ------------------------------------------------------------
# READ APRIL
# ------------------------------------------------------------

future_df_raw = con.execute(
    f"""
    SELECT *
    FROM read_parquet('{april_file}')
    """
).df()

print("\nApril rows:", len(future_df_raw))
print("April columns:", len(future_df_raw.columns))

display(future_df_raw.head())

March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March rows: 9841378
March columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
### Answer

I will use a Random Forest classifier for the Refresh / Content Opportunity Scoring lane.

The decision is which pages should receive review attention first. The model will use March 2026 page-level search and engagement signals to classify whether a page meets the defined future review-outcome label.

Random Forest fits this problem because the relationship between visibility, CTR, position, engagement, and later movement may be non-linear and may involve interactions between signals. It also provides feature importance that can help explain what the model is using.

I will compare the model with my Week-4 rule-based baseline on the same eligible pages and the same evaluation period. The baseline uses visibility and CTR relative to position, while the model can combine several observed signals.

The model is being used for decision support and ranking. A positive prediction does not mean that a refresh will definitely improve the page.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score
)

print("Method: Random Forest Classifier")
print("Task: Refresh / Content Opportunity Scoring")
print("Purpose: rank pages for review using observed pre-outcome signals.")

Method: Random Forest Classifier
Task: Refresh / Content Opportunity Scoring
Purpose: rank pages for review using observed pre-outcome signals.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Answer

I will use a time-aware split.

March 2026 is the feature and decision window. The target is measured from a later outcome window, so the target period is kept separate from the feature period.

I will also keep clients grouped when creating the train/test split so pages from the same client are not split across training and evaluation in a way that makes the evaluation overly optimistic.

The model will only receive information that would have been available at the March decision moment. Future-window performance metrics are used only to construct the target, never as model features.

This makes the evaluation closer to the real decision: use information available at the decision time to identify pages that later meet the review outcome.

In [2]:
print("FEATURE WINDOW: 2026-03")
print("TARGET WINDOW: future period after 2026-03")
print("SPLIT DESIGN: time-aware with client grouping")
print("Leakage rule: future outcome fields are used only to create y, never X.")

FEATURE WINDOW: 2026-03
TARGET WINDOW: future period after 2026-03
SPLIT DESIGN: time-aware with client grouping
Leakage rule: future outcome fields are used only to create y, never X.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Answer

I will train the Random Forest using only features available during the March 2026 decision window.

The model will be compared with the Week-4 baseline using the same eligible page population and the same future outcome label.

The primary ranking metric will be Average Precision because the practical task is to identify a relatively limited set of pages for review. Precision@K will also be reported because editors would normally review only the highest-ranked pages first.

The comparison will show whether the model provides additional ranking signal beyond the transparent Week-4 rule. A higher score will not by itself prove that the model causes better refresh outcomes.

In [1]:
# ============================================================
# ML-08 SECTION 3 — COMPLETE SELF-CONTAINED RAM-SAFE VERSION
# ============================================================

!pip -q install -U huggingface_hub duckdb

import os
import duckdb
import numpy as np
import pandas as pd

from huggingface_hub import login, hf_hub_download
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score


# ============================================================
# 1. HUGGING FACE AUTHENTICATION
# ============================================================

print("Authenticating with Hugging Face...")

login()

print("Hugging Face authentication complete.")


# ============================================================
# 2. DOWNLOAD ONLY THE TWO REQUIRED MONTHS
# ============================================================

print("\nDownloading March 2026...")
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

print("Downloading April 2026...")
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset"
)

print("\nFiles ready.")
print("March:", march_file)
print("April:", april_file)


# ============================================================
# 3. DUCKDB
# ============================================================

con = duckdb.connect()

os.makedirs("work/outputs", exist_ok=True)

print("\nDuckDB connected successfully.")


# ============================================================
# 4. MARCH PAGE-LEVEL FEATURES
# IMPORTANT:
# Aggregate inside DuckDB.
# DO NOT load daily fact rows into pandas.
# ============================================================

print("\nBuilding March page-level dataset...")

page_df = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        SUM(sessions_organic) AS sessions_organic,
        SUM(sessions_direct) AS sessions_direct,
        SUM(sessions_referral) AS sessions_referral,
        SUM(sessions_social) AS sessions_social,
        SUM(sessions_paid) AS sessions_paid,
        SUM(sessions_ai) AS sessions_ai,

        SUM(scroll_events) AS scroll_events

    FROM read_parquet('{march_file}')
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print("March page-level rows:", len(page_df))


# ============================================================
# 5. MARCH FEATURES
# ============================================================

page_df["ctr"] = np.where(
    page_df["gsc_impressions"] > 0,
    page_df["gsc_clicks"] / page_df["gsc_impressions"],
    0
)

page_df["engagement_rate"] = np.where(
    page_df["ga4_sessions"] > 0,
    page_df["ga4_engaged_sessions"] /
    page_df["ga4_sessions"],
    0
)

page_df["avg_engagement_sec"] = np.where(
    page_df["ga4_users"] > 0,
    page_df["ga4_total_engagement_sec"] /
    page_df["ga4_users"],
    0
)

total_sessions = (
    page_df["sessions_organic"]
    + page_df["sessions_direct"]
    + page_df["sessions_referral"]
    + page_df["sessions_social"]
    + page_df["sessions_paid"]
    + page_df["sessions_ai"]
)

page_df["organic_share"] = np.where(
    total_sessions > 0,
    page_df["sessions_organic"] / total_sessions,
    0
)


# ============================================================
# 6. WEEK-4 BASELINE
# ============================================================

def position_tier(position):
    if pd.isna(position):
        return "unknown"
    if position <= 3:
        return "1-3"
    if position <= 10:
        return "4-10"
    if position <= 20:
        return "11-20"
    if position <= 50:
        return "21-50"
    return "51+"


page_df["position_tier"] = (
    page_df["gsc_avg_position"].apply(position_tier)
)

expected_ctr_map = {
    "1-3": 0.0124,
    "4-10": 0.0049,
    "11-20": 0.0032,
    "21-50": 0.0023,
    "51+": 0.0009,
    "unknown": 0.0
}

page_df["expected_ctr"] = (
    page_df["position_tier"]
    .map(expected_ctr_map)
    .fillna(0)
)

page_df["ctr_gap"] = (
    page_df["expected_ctr"] - page_df["ctr"]
).clip(lower=0)

page_df["visibility_score"] = np.log1p(
    page_df["gsc_impressions"]
)

page_df["baseline_score"] = (
    page_df["visibility_score"] *
    page_df["ctr_gap"]
)


# ============================================================
# 7. APRIL FUTURE OUTCOME
# ============================================================

print("\nBuilding April future outcome...")

future_df = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks

    FROM read_parquet('{april_file}')
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

future_df["future_ctr"] = np.where(
    future_df["future_impressions"] > 0,
    future_df["future_clicks"] /
    future_df["future_impressions"],
    0
)

print("April page-level rows:", len(future_df))


# ============================================================
# 8. JOIN MARCH + APRIL
# ============================================================

model_df = page_df.merge(
    future_df,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("Eligible modeling pages:", len(model_df))


# ============================================================
# 9. FUTURE TARGET
# ============================================================

model_df["y"] = (
    model_df["future_ctr"] >
    model_df["expected_ctr"]
).astype(int)

print("\nTARGET DISTRIBUTION")

display(
    model_df["y"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="n")
)

print(
    "Positive rate:",
    round(model_df["y"].mean(), 4)
)


# ============================================================
# 10. FEATURES
# ============================================================

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
    "ctr",
    "engagement_rate",
    "avg_engagement_sec",
    "organic_share",
    "expected_ctr",
    "ctr_gap",
    "visibility_score"
]

X = (
    model_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = model_df["y"].copy()

baseline_score = model_df["baseline_score"].copy()


# ============================================================
# 11. CLIENT-GROUPED SPLIT
# ============================================================

unique_clients = (
    model_df["client_hash_id"]
    .drop_duplicates()
    .tolist()
)

rng = np.random.RandomState(42)
rng.shuffle(unique_clients)

split_point = int(len(unique_clients) * 0.80)

train_clients = set(
    unique_clients[:split_point]
)

test_clients = set(
    unique_clients[split_point:]
)

train_mask = model_df["client_hash_id"].isin(
    train_clients
)

test_mask = model_df["client_hash_id"].isin(
    test_clients
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

baseline_score_train = (
    baseline_score.loc[train_mask].copy()
)

baseline_score_test = (
    baseline_score.loc[test_mask].copy()
)

print("\nSPLIT")
print("=" * 50)
print("Train pages   :", len(X_train))
print("Test pages    :", len(X_test))
print("Train clients :", len(train_clients))
print("Test clients  :", len(test_clients))
print("Train positive:", round(y_train.mean(), 4))
print("Test positive :", round(y_test.mean(), 4))


# ============================================================
# 12. RANDOM FOREST
# ============================================================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

print("\nMODEL TRAINED SUCCESSFULLY")


# ============================================================
# 13. PREDICTIONS
# ============================================================

model_score_test = (
    model.predict_proba(X_test)[:, 1]
)


# ============================================================
# 14. PRECISION@K
# ============================================================

def precision_at_k(y_true, scores, k=50):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    order = np.argsort(scores)[::-1][:k]

    return y_true[order].mean()


# ============================================================
# 15. MODEL METRICS
# ============================================================

model_ap = average_precision_score(
    y_test,
    model_score_test
)

model_auc = roc_auc_score(
    y_test,
    model_score_test
)

model_p50 = precision_at_k(
    y_test,
    model_score_test,
    50
)


# ============================================================
# 16. BASELINE METRICS
# ============================================================

baseline_ap = average_precision_score(
    y_test,
    baseline_score_test
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_score_test
)

baseline_p50 = precision_at_k(
    y_test,
    baseline_score_test,
    50
)


# ============================================================
# 17. COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "average_precision": [
        baseline_ap,
        model_ap
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

print("\nMODEL VS WEEK-4 BASELINE")

display(comparison)


# ============================================================
# 18. SAVE
# ============================================================

comparison.to_csv(
    "work/outputs/model_vs_baseline.csv",
    index=False
)

print("\nSaved:")
print("work/outputs/model_vs_baseline.csv")

print("\nSECTION 3 COMPLETE.")

Authenticating with Hugging Face...
Hugging Face authentication complete.


Files ready.
March: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet

DuckDB connected successfully.

Building March page-level dataset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March page-level rows: 331437

Building April future outcome...
April page-level rows: 362172
Eligible modeling pages: 331436

TARGET DISTRIBUTION


,target,n
0,0,309269
1,1,22167


Positive rate: 0.0669

SPLIT
Train pages   : 245483
Test pages    : 85953
Train clients : 44
Test clients  : 11
Train positive: 0.0649
Test positive : 0.0727

MODEL TRAINED SUCCESSFULLY

MODEL VS WEEK-4 BASELINE


,method,average_precision,roc_auc,precision_at_50
0,Week-4 baseline,0.074192,0.490413,0.00
1,Random Forest,0.386598,0.811229,0.94



Saved:
work/outputs/model_vs_baseline.csv

SECTION 3 COMPLETE.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Answer

I inspected false positives and false negatives rather than relying only on the overall metric. False positives are pages the model ranks as likely positive but whose observed outcome is negative; false negatives are pages with the observed positive outcome that receive a lower model score.

I also inspected the model's feature importance to understand which available signals the model relied on most. These importances describe associations used by the fitted model; they do not establish causal effects.

The error review is used as decision-support: a high model score is a reason to review a page, not a guarantee that an action will improve performance.

In [6]:
# ============================================================
# 4. ERRORS AND INTERPRETATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# A. Build error-analysis dataframe
# ------------------------------------------------------------

error_analysis = X_test.copy().reset_index(drop=True)

error_analysis["actual"] = np.asarray(y_test)
error_analysis["model_score"] = np.asarray(model_score_test)
error_analysis["baseline_score"] = np.asarray(baseline_score_test)

# Model classification using the default 0.50 threshold
error_analysis["model_prediction"] = (
    error_analysis["model_score"] >= 0.50
).astype(int)

# ------------------------------------------------------------
# B. Identify the four types of model outcomes
# ------------------------------------------------------------

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["model_prediction"] == 1),

        (error_analysis["actual"] == 0) &
        (error_analysis["model_prediction"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["model_prediction"] == 1),

        (error_analysis["actual"] == 1) &
        (error_analysis["model_prediction"] == 0),
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative",
    ],
    default="Unknown"
)

print("MODEL OUTCOME COUNTS")
display(
    error_analysis["error_type"]
    .value_counts()
    .rename_axis("outcome")
    .reset_index(name="n")
)


# ------------------------------------------------------------
# C. Inspect the highest-confidence false positives
# ------------------------------------------------------------

false_positives = (
    error_analysis[
        error_analysis["error_type"] == "False Positive"
    ]
    .sort_values("model_score", ascending=False)
)

print("\nTOP FALSE POSITIVES")
print("These are pages the model scored highly but the observed target was negative.")

display(
    false_positives.head(10)
)


# ------------------------------------------------------------
# D. Inspect the highest-confidence false negatives
# ------------------------------------------------------------

false_negatives = (
    error_analysis[
        error_analysis["error_type"] == "False Negative"
    ]
    .sort_values("model_score", ascending=True)
)

print("\nTOP FALSE NEGATIVES")
print("These are pages with a positive observed target that received a relatively low model score.")

display(
    false_negatives.head(10)
)


# ------------------------------------------------------------
# E. Compare the model's top-ranked pages with the baseline
# ------------------------------------------------------------

top_k = min(20, len(error_analysis))

model_top = (
    error_analysis
    .sort_values("model_score", ascending=False)
    .head(top_k)
)

baseline_top = (
    error_analysis
    .sort_values("baseline_score", ascending=False)
    .head(top_k)
)

model_top_precision = model_top["actual"].mean()
baseline_top_precision = baseline_top["actual"].mean()

print("\nTOP-20 ERROR CHECK")
print(f"Model top-{top_k} observed positive rate:    {model_top_precision:.4f}")
print(f"Baseline top-{top_k} observed positive rate: {baseline_top_precision:.4f}")


# ------------------------------------------------------------
# F. Feature importance
# ------------------------------------------------------------

if hasattr(model, "feature_importances_"):

    importance = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_
    })

    importance = (
        importance
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    print("\nTOP MODEL FEATURES")
    display(importance.head(10))

else:
    print("\nThis model does not provide feature_importances_.")


# ------------------------------------------------------------
# G. Simple error rates
# ------------------------------------------------------------

tp = (
    (error_analysis["actual"] == 1) &
    (error_analysis["model_prediction"] == 1)
).sum()

tn = (
    (error_analysis["actual"] == 0) &
    (error_analysis["model_prediction"] == 0)
).sum()

fp = (
    (error_analysis["actual"] == 0) &
    (error_analysis["model_prediction"] == 1)
).sum()

fn = (
    (error_analysis["actual"] == 1) &
    (error_analysis["model_prediction"] == 0)
).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

error_summary = pd.DataFrame({
    "metric": [
        "True positives",
        "True negatives",
        "False positives",
        "False negatives",
        "Precision",
        "Recall"
    ],
    "value": [
        tp,
        tn,
        fp,
        fn,
        precision,
        recall
    ]
})

print("\nERROR SUMMARY")
display(error_summary)


# ------------------------------------------------------------
# H. Save error-analysis outputs
# ------------------------------------------------------------

error_summary.to_csv(
    "work/outputs/model_error_summary.csv",
    index=False
)

if hasattr(model, "feature_importances_"):
    importance.to_csv(
        "work/outputs/model_feature_importance.csv",
        index=False
    )

print("\nSaved error-analysis outputs.")

MODEL OUTCOME COUNTS


,outcome,n
0,True Negative,65531
1,False Positive,14177
2,True Positive,4155
3,False Negative,2090



TOP FALSE POSITIVES
These are pages the model scored highly but the observed target was negative.


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,...,avg_engagement_sec,organic_share,expected_ctr,ctr_gap,visibility_score,actual,model_score,baseline_score,model_prediction,error_type
18664,12608.0,205.0,10.874475,205.0,152.0,147.0,3.0,717.0,211.0,5.0,...,4.877551,0.967890,0.0032,0.0,9.442166,0,0.978645,0.0,1,False Positive
45836,10854.0,137.0,14.372791,146.0,129.0,127.0,4.0,215.0,179.0,8.0,...,1.692913,0.957219,0.0032,0.0,9.292381,0,0.977780,0.0,1,False Positive
35549,4520.0,168.0,8.443891,157.0,146.0,143.0,11.0,257.0,183.0,14.0,...,1.797203,0.928934,0.0049,0.0,8.416488,0,0.975176,0.0,1,False Positive
18802,16349.0,379.0,6.051372,467.0,423.0,385.0,17.0,1723.0,609.0,48.0,...,4.475325,0.876259,0.0049,0.0,9.701983,0,0.970314,0.0,1,False Positive
32059,17295.0,166.0,8.692787,161.0,130.0,129.0,13.0,3307.0,155.0,39.0,...,25.635659,0.790816,0.0049,0.0,9.758231,0,0.969955,0.0,1,False Positive
65148,25208.0,203.0,7.926523,203.0,176.0,171.0,9.0,1622.0,247.0,13.0,...,9.485380,0.942748,0.0049,0.0,10.134956,0,0.969088,0.0,1,False Positive
18365,13987.0,102.0,9.045735,108.0,90.0,89.0,4.0,356.0,124.0,9.0,...,4.000000,0.932331,0.0049,0.0,9.545955,0,0.969075,0.0,1,False Positive
18622,9869.0,67.0,12.481047,73.0,65.0,64.0,5.0,2475.0,83.0,7.0,...,38.671875,0.922222,0.0032,0.0,9.197255,0,0.968508,0.0,1,False Positive
31086,27564.0,337.0,5.212509,272.0,261.0,256.0,11.0,502.0,373.0,14.0,...,1.960938,0.963824,0.0049,0.0,10.224302,0,0.966853,0.0,1,False Positive
31508,8609.0,67.0,6.317500,75.0,65.0,64.0,2.0,1705.0,85.0,13.0,...,26.640625,0.867347,0.0049,0.0,9.060680,0,0.965635,0.0,1,False Positive



TOP FALSE NEGATIVES
These are pages with a positive observed target that received a relatively low model score.


...


TOP-20 ERROR CHECK
Model top-20 observed positive rate:    0.9000
Baseline top-20 observed positive rate: 0.0000

TOP MODEL FEATURES


,feature,importance
0,ctr,0.187975
1,visibility_score,0.171141
2,gsc_impressions,0.130018
3,ctr_gap,0.113575
4,gsc_avg_position,0.096014
5,gsc_clicks,0.071851
6,organic_share,0.063429
7,sessions_organic,0.046318
8,expected_ctr,0.038886
9,ga4_pageviews,0.023784



ERROR SUMMARY


,metric,value
0,True positives,4155.000000
1,True negatives,65531.000000
2,False positives,14177.000000
3,False negatives,2090.000000
4,Precision,0.226653
5,Recall,0.665332



Saved error-analysis outputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.